In [1]:
import os
import sys

os.chdir("..")
sys.path.insert(0, "src")

In [2]:
from config import settings
import json
from datetime import datetime, timezone, timedelta
import requests
from bank_client import jwt_gen

In [18]:
API_ORIGIN = "https://api.enablebanking.com"
ASPSP_NAME = "Erste Bank"
ASPSP_COUNTRY = "HU"

with open(settings.sessions_info_path, "r") as f:
    session_info = json.load(f)

base_headers = jwt_gen()

# Using the first available account for the following API calls
account_uid = session_info["Erste Bank"]['accounts'][0]['uid']

all_transactions = []

# Retrieving account balances
r = requests.get(f"{API_ORIGIN}/accounts/{account_uid}/balances", headers=base_headers)
if r.status_code == 200:
    print("Balances:")
    print(r.json())
else:
    print(f"Error response {r.status_code}:", r.text)


# Retrieving account transactions (since 90 days ago)
query = {
    "date_from": (datetime.now(timezone.utc) - timedelta(days=90)).date().isoformat(),
}
continuation_key = None
while True:
    if continuation_key:
        query["continuation_key"] = continuation_key
    r = requests.get(
        f"{API_ORIGIN}/accounts/{account_uid}/transactions",
        params=query,
        headers=base_headers,
    )
    if r.status_code == 200:
        resp_data = r.json()
        all_transactions.extend(resp_data["transactions"])
        continuation_key = resp_data.get("continuation_key")
        if not continuation_key:
            print("No continuation key. All transactions were fetched")
            break
        print(f"Going to fetch more transactions with continuation key {continuation_key}")
    else:
        print(f"Error response {r.status_code}:", r.text)

with open("./data/transactions_erste_bank.json", "w") as f:
    json.dump(all_transactions, f, indent=2)

print("All done!")

Balances:
{'balances': [{'name': 'Interim booked balance', 'balance_amount': {'currency': 'HUF', 'amount': '-1673.00'}, 'balance_type': 'ITBD', 'last_change_date_time': '2026-06-22T12:27:08+02:00', 'reference_date': None, 'last_committed_transaction': None}, {'name': 'Interim available balance', 'balance_amount': {'currency': 'HUF', 'amount': '-1673.00'}, 'balance_type': 'ITAV', 'last_change_date_time': '2026-06-22T12:27:08+02:00', 'reference_date': None, 'last_committed_transaction': None}, {'name': 'Balance of type "BLOCKED"', 'balance_amount': {'currency': 'HUF', 'amount': '0.00'}, 'balance_type': 'OTHR', 'last_change_date_time': '2026-06-22T12:27:08+02:00', 'reference_date': None, 'last_committed_transaction': None}]}
No continuation key. All transactions were fetched
All done!


In [87]:
with get_session() as session:
    test = session.query(Balance.bank_name, Balance.currency, Balance.balance_type, Balance.amount, Balance.retrieved_at).all()
    session.expunge_all()

In [ ]:
(test[0].retrieved_at)

TypeError: 'datetime.datetime' object cannot be interpreted as an integer

In [3]:
from mcp_server import get_balance

test = get_balance()

In [4]:
test

{('Revolut', 'EUR'): (0.0, datetime.datetime(2026, 6, 23, 17, 59, 4, 107266)),
 ('Erste Bank', 'HUF'): (-1673.0,
  datetime.datetime(2026, 6, 23, 17, 59, 4, 103797))}

In [24]:
test = {('asd', 'efg'): 3,
        ('idk', 'nemtom'): 4}

In [23]:
[key[0] for key in test.keys()]

['asd']

In [27]:
for key, value in test.items():
    print(key[0])

asd
idk
